In [5]:
!pip install  langchain langchain-ollama langchain-core duckduckgo-search

In [6]:
import requests
import os
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from dotenv import load_dotenv
load_dotenv()


True

In [7]:
try:
    api_key=os.getenv("OPENROUTER_API_KEY")
    api_key2=os.getenv('TAVILY_API_KEY')
    api_key3=os.getenv('WEATHERSTACK_API_KEY')
    print('Fetched')
except Exception as e:
    print('Error is ',e) 

Fetched


In [8]:
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.tools.tavily_search import TavilySearchResults
search_tool1=TavilySearchResults(max_results=3,api_key=api_key2,description="A high-performance, structured search engine. USE THIS TOOL FIRST for all factual queries, esports data, player stats, and multi-step lookups.")
search_tool2=DuckDuckGoSearchRun()


/var/folders/z8/x7qpgkld3gg8x0_qgy81tmjc0000gn/T/ipykernel_16427/1843044077.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun
/var/folders/z8/x7qpgkld3gg8x0_qgy81tmjc0000gn/T/ipykernel_16427/1843044077.py:3: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_tool1=TavilySearchResults(max_results=3,api_key=api_key2,description="A high-performance, structured search engine. USE THIS TOOL FIRST for all factual queries, esports data, player stats, and mu

In [20]:
@tool

def get_weather_details(city:str) -> str:
    """This function give the details of the weather of provided city"""
    try:
        api_key=os.getenv('WEATHERSTACK_API_KEY')
    except Exception as e:
        print('Error is ',e) 
        
    url=f"https://api.weatherstack.com/current?access_key={api_key}&query={city}"
    response=requests.get(url)
    
    return response.json()

In [13]:
from langchain_openai import ChatOpenAI

# Point to Ollama's cloud endpoint using your Ollama API key
llm = ChatOpenAI(
    model="qwen/qwen-2.5-7b-instruct",  # or whichever Qwen model ID OpenRouter lists
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key=api_key,
    temperature=0
)

# llm=ChatOllama(
#     model='qwen2.5:3b-instruct'
# )


In [14]:
from langchain_classic.agents import create_react_agent,AgentExecutor
from langsmith import Client

/Users/pratiksha/Documents/PYTHON__XHIS_XHIS/Gen_Ai/nwee/helloo_there/samrat/Samrat/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
client=Client()
try:
    prompt=client.pull_prompt('hwchase17/react',dangerously_pull_public_prompt=True) # pulls the standard React agent prompt
except Exception as e:
    print('Error is ',e) 

In [23]:
#create the react agent manaually with the pulled prompt
agent1=create_react_agent(
    llm=llm,
    tools=[search_tool1,get_weather_details],
    prompt=prompt
)

In [25]:
#wrap it with agentexecutor
agent_executor=AgentExecutor(
    agent=agent1,
    tools=[search_tool1,get_weather_details],
    verbose=True
)

In [22]:
react_prompt = f"What is the capital city of nepal and what its weather condition"

In [26]:
response = agent_executor.invoke({
    "input":react_prompt
})




> Entering new AgentExecutor chain...
I should first find out the capital city of Nepal using the search tool, and then check its weather condition.
Action: tavily_search_results_json
Action Input: {"query": "capital city of nepal"}[{'title': 'The Nepal Capital | TravellersQuest', 'url': 'https://www.travellersquest.com/blog/nepal-capital', 'content': 'Home » blog » Capital City Of Nepal\n\n# Capital City Of Nepal\n\nkathmandu   \n\nImage by willthomas from Pixabay\n\nThinking of traveling to Nepal? Are you interested to learn detailed information about Nepal’s capital? Do you wish to know what are the favorite things to do in the capital of Nepal? You have come to the right place!\n\nThis article will help you understand all the details regarding the capital city of Nepal, including the best things to do, top hotels to stay, top restaurants to dine at, top sights to visit and much more!\n\nKathmandu is the capital city of Nepal and is located between two major cities: Bhaktapur and 

In [27]:
response['output']

'The capital city of Nepal is Kathmandu. The current weather in Kathmandu is light rain showers with a temperature of 16°C, humidity at 95%, and cloud cover of 100%.'

In [28]:
from langchain_experimental.plan_and_execute import (
    PlanAndExecute,
    load_agent_executor,
    load_chat_planner
)

/var/folders/z8/x7qpgkld3gg8x0_qgy81tmjc0000gn/T/ipykernel_16427/2591945707.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.plan_and_execute import (


In [29]:
tools=[search_tool1]

In [30]:
planner=load_chat_planner(llm)
executor=load_agent_executor(llm,tools)

In [31]:
agent=PlanAndExecute(
    planner=planner,
    executor=executor,
    verbose=True
)

In [32]:
plan_prompt = (
   "How do we sum numbers in c program"
)


In [ ]:
response=agent.invoke({
    "input": plan_prompt
})

In [ ]:
response

In [ ]:
response['output']